# Gradient Boosting vs Random Forest

In [64]:
import pandas as pd

df = pd.read_csv("employee_promotion_5000.csv")

X = df[
    [
        "Age",
        "Experience",
        "Monthly_Bonus",
        "Department",
        "Education"
    ]
]

y = df["Promoted"]

In [65]:
numeric_features = [
    "Age",
    "Experience",
    "Monthly_Bonus",
]

categorical_features = [
    "Department",
    "Education"
]

In [66]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer


numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_pipeline,
            numeric_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ]
)

In [67]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [68]:
from sklearn.ensemble import (
    GradientBoostingClassifier,
    RandomForestClassifier
)

models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )
}

In [69]:
pipelines = {}

for name, model in models.items():
    pipelines[name] = Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor
            ),
            (
                "model",
                model
            )
        ]
    )

In [70]:
for name, pipeline in pipelines.items():
    pipeline.fit(
        X_train,
        y_train
    )

    print(name, "Trained")

Random Forest Trained
Gradient Boosting Trained


In [71]:
predictions = {}

for name, pipeline in pipelines.items():
    predictions[name] = pipeline.predict(X_test)

In [72]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

evaluation_results = []

for name, prediction in predictions.items():

    evaluation_results.append({
        "Model": name,
        
        "Accuracy": accuracy_score(
            y_test,
            prediction
        ),

        "Precision": precision_score(
            y_test,
            prediction,
            zero_division=0
        ),

        "Recall": recall_score(
            y_test,
            prediction,
            zero_division=0
        ),

        "F1": f1_score(
            y_test,
            prediction,
            zero_division=0
        )
    })

In [73]:
evaluation_df = pd.DataFrame(
    evaluation_results
)

print(
    evaluation_df.sort_values(
        "F1",
        ascending=False
    )
)

               Model  Accuracy  Precision    Recall        F1
0      Random Forest     0.828   0.396226  0.130435  0.196262
1  Gradient Boosting     0.832   0.294118  0.031056  0.056180


In [74]:
from sklearn.model_selection import cross_val_score

cv_results = []

for name, pipeline in pipelines.items():

    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=5,
        scoring="f1"
    )

    cv_results.append({
        "Model": name,
        "Mean_CV_F1": scores.mean(),
        "Std_CV_F1": scores.std()
    })


cv_df = pd.DataFrame(cv_results)

print(
    cv_df.sort_values(
        "Mean_CV_F1",
        ascending=False
    )
)

               Model  Mean_CV_F1  Std_CV_F1
0      Random Forest    0.188747   0.017624
1  Gradient Boosting    0.140574   0.041685


In [75]:
final_comparison = evaluation_df.merge(
    cv_df,
    on="Model"
)

final_comparison = final_comparison.sort_values(
    "Mean_CV_F1",
    ascending=False
)

print(final_comparison.round(3))

               Model  Accuracy  Precision  Recall     F1  Mean_CV_F1  \
0      Random Forest     0.828      0.396   0.130  0.196       0.189   
1  Gradient Boosting     0.832      0.294   0.031  0.056       0.141   

   Std_CV_F1  
0      0.018  
1      0.042  


In [76]:
train_results = []

for name, pipeline in pipelines.items():

    train_prediction = pipeline.predict(X_train)

    train_results.append({
        "Model": name,
        "Train_F1": f1_score(
            y_train,
            train_prediction,
            zero_division=0
        )
    })

train_df = pd.DataFrame(
    train_results
)

final_comparison = final_comparison.merge(
    train_df,
    on="Model"
)

In [77]:
final_comparison = final_comparison[
    [
        "Model",
        "Train_F1",
        "F1",
        "Mean_CV_F1",
        "Std_CV_F1",
        "Accuracy",
        "Precision",
        "Recall"
    ]
]

print(
    final_comparison.round(3)
)

               Model  Train_F1     F1  Mean_CV_F1  Std_CV_F1  Accuracy  \
0      Random Forest     0.999  0.196       0.189      0.018     0.828   
1  Gradient Boosting     0.238  0.056       0.141      0.042     0.832   

   Precision  Recall  
0      0.396   0.130  
1      0.294   0.031  
